# 04 K-Center TF-IDF Baseline

Build TF-IDF features for AG News train texts, select `K_PER_CLASS` k-center examples per class, then train the classifier on the selected coreset.

In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from text_distillation.data import get_dataset_info, get_train_eval_splits, load_text_classification_dataset, make_tiny_subset
from text_distillation.data.datasets import get_label_names
from text_distillation.distillation import select_kcenter_tfidf
from text_distillation.model import train_text_classifier
from text_distillation.saving import create_run_dir, save_distilled_dataset, save_experiment_config, save_metrics
from text_distillation.utils import get_git_commit_hash, set_seed

In [ ]:
EXPERIMENT_NAME = "kcenter_tfidf_agnews_bert_base_k100"
DATASET_NAME = "ag_news"
METHOD_NAME = "kcenter_tfidf"
MODEL_NAME = "bert-base-uncased"
# Supported datasets: ag_news, sst2, qqp, mnli-m
DATASET_INFO = get_dataset_info(DATASET_NAME)
TEXT_COLUMNS = DATASET_INFO.text_columns
LABEL_COLUMN = DATASET_INFO.label_column
METRIC_NAME = DATASET_INFO.metric_name
K_PER_CLASS = 100
TFIDF_MAX_FEATURES = 50_000
MAX_LENGTH = 128
NUM_TRAIN_EPOCHS = 5.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
# T4 16GB starting point. If CUDA OOM happens, lower TRAIN_BATCH_SIZE to 32.
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1
MIXED_PRECISION = "auto"
DATALOADER_NUM_WORKERS = 2
SEED = 42

# Set for smoke checks before full runs.
MAX_TRAIN_POOL_SAMPLES = None
MAX_EVAL_SAMPLES = None

set_seed(SEED)

In [ ]:
dataset = load_text_classification_dataset(DATASET_NAME)
train_dataset, test_dataset = get_train_eval_splits(dataset, DATASET_NAME)

if MAX_TRAIN_POOL_SAMPLES is not None:
    train_dataset = make_tiny_subset(train_dataset, total_size=MAX_TRAIN_POOL_SAMPLES, seed=SEED)
if MAX_EVAL_SAMPLES is not None:
    test_dataset = make_tiny_subset(test_dataset, total_size=MAX_EVAL_SAMPLES, seed=SEED)

selection_start = time.perf_counter()
distilled_dataset = select_kcenter_tfidf(
    dataset=train_dataset,
    text_columns=TEXT_COLUMNS,
    label_column=LABEL_COLUMN,
    k_per_class=K_PER_CLASS,
    seed=SEED,
    max_features=TFIDF_MAX_FEATURES,
)
selection_time_sec = time.perf_counter() - selection_start

label_names = get_label_names(train_dataset, LABEL_COLUMN)
num_labels = len(label_names) if label_names is not None else len(set(train_dataset[LABEL_COLUMN]))
print(distilled_dataset)
print(test_dataset)
print("selection_time_sec", selection_time_sec)

In [ ]:
run_dir = create_run_dir(PROJECT_ROOT / "artifacts" / "runs", EXPERIMENT_NAME)
config = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_name": DATASET_NAME,
    "method_name": METHOD_NAME,
    "model_name": MODEL_NAME,
    "text_columns": TEXT_COLUMNS,
    "label_column": LABEL_COLUMN,
    "metric_name": METRIC_NAME,
    "train_split": DATASET_INFO.train_split,
    "eval_split": DATASET_INFO.eval_split,
    "k_per_class": K_PER_CLASS,
    "k_total": len(distilled_dataset),
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "max_length": MAX_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mixed_precision": MIXED_PRECISION,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "seed": SEED,
    "full_train_size": len(dataset[DATASET_INFO.train_split]),
    "train_pool_size": len(train_dataset),
    "train_size": len(distilled_dataset),
    "eval_size": len(test_dataset),
    "compression_ratio": len(dataset[DATASET_INFO.train_split]) / len(distilled_dataset),
    "selection_time_sec": selection_time_sec,
    "git_commit": get_git_commit_hash(PROJECT_ROOT),
}
save_experiment_config(config, run_dir)
save_distilled_dataset(distilled_dataset, run_dir)

In [ ]:
train_start = time.perf_counter()
model, metrics = train_text_classifier(
    train_dataset=distilled_dataset,
    eval_dataset=test_dataset,
    model_name=MODEL_NAME,
    output_dir=run_dir,
    text_columns=TEXT_COLUMNS,
    label_column=LABEL_COLUMN,
    num_labels=num_labels,
    label_names=label_names,
    max_length=MAX_LENGTH,
    metric_name=METRIC_NAME,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    mixed_precision=MIXED_PRECISION,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    seed=SEED,
)
metrics["training_time_sec"] = time.perf_counter() - train_start
metrics["selection_time_sec"] = selection_time_sec
save_metrics(metrics, run_dir)
metrics